# Day 15 - Diffusion: Marketing Imagery

## RetailMind AI Increment

Day 15 adds a creative-generation module to RetailMind AI. Instead of training on a new dataset, the workflow uses a pretrained Stable Diffusion model and builds controlled marketing prompts from local product metadata.

### Business objective
Marketing teams often need multiple campaign visuals for the same product. A reproducible diffusion workflow can generate draft hero, social, and catalog concepts quickly, while seeds, prompt templates, negative prompts, and human approval help preserve consistency.

## Dataset and runtime

**Dataset:** none for model training. Stable Diffusion is already pretrained. RetailMind supplies only local product metadata such as product name, category, material, brand colors, visual style, and setting.

Expected file:

```text
data/product_marketing_metadata.json
```

Generation requires a CUDA GPU. The pretrained `runwayml/stable-diffusion-v1-5` weights must be available from the Hugging Face Hub or already cached in the environment.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))
import day15_diffusion_marketing_imagery as day15

print("Model:", day15.MODEL_ID)
print("Image size:", day15.DEFAULT_WIDTH, "x", day15.DEFAULT_HEIGHT)
print("Guidance scale:", day15.DEFAULT_GUIDANCE)
print("Inference steps:", day15.DEFAULT_STEPS)

## 1. Product metadata becomes the prompt source

A structured prompt is safer and more repeatable than manually inventing every prompt. Product metadata controls category, material, palette, campaign type, setting, and brand style.

In [ ]:
# Example metadata if the real JSON has not yet been prepared.
example_product = {
    "product_id": "RM-HEADPHONE-001",
    "product_name": "Nimbus Wireless Headphones",
    "category": "wireless headphones",
    "material": "matte aluminium and soft black ear cushions",
    "brand_colors": "deep navy, silver, subtle teal accents",
    "brand_style": "premium minimal technology campaign, clean and trustworthy",
    "setting": "modern studio with soft gradient background",
}

for campaign in ["hero", "social", "catalog"]:
    print("
", campaign.upper())
    print(day15.build_prompt(example_product, campaign))

## 2. Negative prompts and guidance

Negative prompts discourage unwanted properties such as distorted geometry, accidental text, watermarks, extra products, cropping, and poor quality. Guidance scale controls how strongly the generated image follows the text prompt. The default `7.5` is a practical starting point, not a guarantee of brand correctness.

In [ ]:
print(day15.build_negative_prompt(example_product))

for guidance in [5.5, 7.5, 9.0]:
    print(f"guidance={guidance}: lower values allow more freedom; higher values push prompt adherence")

## 3. Load Stable Diffusion on GPU

The project uses `DPMSolverMultistepScheduler` for a practical inference schedule. `float16` reduces GPU memory use. Generation is intentionally separated from repository setup so source code can be versioned even when the hosted workspace lacks a compatible GPU.

In [ ]:
# Run only in a CUDA-enabled environment with model access/cache.
# pipe = day15.load_pipeline(local_files_only=False)
# print("Pipeline device:", pipe.device)

## 4. Reproducible generation with fixed seeds

A seed is part of the asset's audit record. The same model version, prompt, negative prompt, scheduler settings, dimensions, steps, guidance scale, and seed are needed for reproducibility.

In [ ]:
seeds = [42, 108, 2026]
for seed in seeds:
    print(seed, day15.build_prompt(example_product, "hero")[:110] + "...")

# Example generation:
# image, metadata = day15.generate_asset(pipe, example_product, seed=42, campaign="hero")
# display(image)
# metadata

## 5. Compare prompt controls

Generate several small controlled variants instead of changing everything at once:

- keep seed fixed and change `campaign` to study composition;
- keep prompt fixed and vary guidance scale to study prompt adherence;
- keep all generation settings fixed and change only the seed to create alternative candidates.

This makes visual review more defensible than random prompt experimentation.

In [ ]:
library = day15.build_prompt_library([example_product])
print("Prompt-library entries:", len(library))
for item in library[:4]:
    print(item["campaign"], item["seed"], item["guidance_scale"])

## 6. Human quality and brand review

Diffusion output must be reviewed before campaign use. Check:

1. product identity and geometry;
2. correct material/color impression;
3. no unwanted text or fake logos;
4. no duplicated or impossible product components;
5. product is not cropped unexpectedly;
6. composition matches the selected campaign use;
7. no misleading claim is visually implied.

Approved assets should retain their generation JSON so creative decisions can be reproduced.

In [ ]:
review = day15.quality_review(example_product)
print(review)
# After visual review, replace None values in the saved JSON with true/false and notes.

## 7. Save generated images and prompt metadata

Each PNG is paired with JSON containing product ID, campaign type, seed, model, prompt, negative prompt, guidance scale, inference steps, and a review block.

In [ ]:
# After generation:
# image_path, json_path = day15.save_asset(image, metadata, repo_root / day15.OUTPUT_DIR)
# print(image_path)
# print(json_path)

## 8. RetailMind creative-module integration

The module should receive approved product metadata rather than arbitrary free-text prompts. A future service can expose campaign type, seed and approved style presets while keeping model parameters and negative-prompt standards controlled. Day 16 can build on these approved images with img2img variations and short promotional-video generation.

### Key limitation
Stable Diffusion can create visually attractive images that do not faithfully reproduce an exact retail SKU. Until stronger product-conditioning or image-to-image techniques are introduced, these outputs should be treated as marketing concepts requiring human approval, not authoritative catalog photographs.